<a href="https://colab.research.google.com/github/yourbells/Gemini-ChatBot---GadGetBot-/blob/main/GadGetBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q streamlit pyngrok google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.4 MB/s eta 0:00:00


In [12]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('API_KEY'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [13]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [19]:
%%writefile streamlit_app_basic.py
import streamlit as st
import pandas as pd
import numpy as np
import time
from google import genai

st.title("GadGetBot")
st.caption("Chatbot sederhana untuk memudahkan kamu memilih Gadget sesuai budGet")

# =========================
# Sidebar Configuration
# =========================
st.sidebar.header("⚙️ Pengaturan Chatbot")

api_key = st.sidebar.text_input(
    "Masukkan Gemini API Key",
    type="password",
    help="API key hanya dipakai saat aplikasi berjalan. Jangan upload API key ke GitHub."
)

model_name = st.sidebar.selectbox(
    "Model",
    ["gemini-3.5-flash", "gemini-2.5-flash"],
    index=0
)

response_style = st.sidebar.selectbox(
    "Gaya Jawaban",
    ["Formal", "Santai", "Ringkas", "Detail"],
    index=0
)

temperature = st.sidebar.slider(
    "Temperature",
    min_value=0.0,
    max_value=1.5,
    value=0.4,
    step=0.1,
    help="Semakin rendah, jawaban lebih stabil. Semakin tinggi, jawaban lebih kreatif."
)

if st.sidebar.button("Reset Chat"):
    st.session_state.messages = []
    st.session_state.previous_interaction_id = None
    st.rerun()

# =========================
# Session State
# =========================
if "messages" not in st.session_state:
    st.session_state.messages = []

if "previous_interaction_id" not in st.session_state:
    st.session_state.previous_interaction_id = None

# Initial assistant message
if len(st.session_state.messages) == 0:
    st.session_state.messages.append({
        "role": "assistant",
        "content": (
            "Halo. Saya GatGetBot. "
            "Saya dapat membantu anda memilih Gadget terbaik sesuai Budget Anda"
        )
    })

system_instruction = f"""
Kamu adalah Seorang Tech Enthusiast yang memiliki banyak pengetahuan seputar Gadget.
Manusia akan bertanya kepadamu tentang rekomendasi gadget yang sesuai dengan budget yang mereka miliki.
Tugasmu adalah sebagai berikut:
1. Jangan keluar dari topik yang sudah ditentukan yaitu seputar Gadget.
2. Tanyakan pada User Gadget apa yang sedang dicari dan berapa Budget yang dimiliki.
3. Membantu User untuk memilih Gadget apa yang sesuai dengan Budget yang mereka miliki,
4. Memberikan rerekomendasikan Gadget yang sesuai dengan Budget yang User miliki.
5. Memberitahukan User apa kelebihan dan kekurangan dari masing masing brand.
6. Jangan memberikan rekomendasi gadget yang tidak sesuai dengan budget yang telah ditentukan.

Gunakan Bahasa Indonesia menggunakan gaya bahasa yang dipilih oleh user: {response_style}

"""

# =========================
# Display Chat History
# =========================
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])


# =========================
# Chat Input
# =========================
user_prompt = st.chat_input("Tuliskan Gadget apa yang sedang kamu cari dan berapa Budget kamu")

if user_prompt:
    st.session_state.messages.append({
        "role": "user",
        "content": user_prompt
    })

    with st.chat_message("user"):
        st.markdown(user_prompt)

    if not api_key:
        warning_message = "Silakan masukkan Gemini API Key terlebih dahulu di sidebar."
        st.session_state.messages.append({
            "role": "assistant",
            "content": warning_message
        })
        with st.chat_message("assistant"):
            st.warning(warning_message)
    else:
        with st.chat_message("assistant"):
            with st.spinner("Harap Bersabar Ini Ujian.."):
                try:
                    client = genai.Client(api_key=api_key)

                    input_text = f"""
User:
{user_prompt}
"""

                    request_params = {
                        "model": model_name,
                        "system_instruction": system_instruction,
                        "input": input_text,
                        "generation_config": {
                            "temperature": temperature
                        }
                    }

                    if st.session_state.previous_interaction_id:
                        request_params["previous_interaction_id"] = st.session_state.previous_interaction_id

                    interaction = client.interactions.create(**request_params)

                    bot_response = interaction.output_text
                    st.session_state.previous_interaction_id = interaction.id

                    st.markdown(bot_response)

                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": bot_response
                    })

                except Exception as e:
                    error_message = f"Terjadi error: {e}"
                    st.error(error_message)
                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": error_message
                    })


Overwriting streamlit_app_basic.py


In [20]:
proc = run_streamlit("streamlit_app_basic.py")

Streamlit berjalan: NgrokTunnel: "https://distant-consonant-mushily.ngrok-free.dev" -> "http://localhost:8501"


In [21]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.
